# Universidade Katyavala Bwila — Instituto Politécnico
## Mestrado em Engenharia Informática — Aprendizagem Profunda (2025/2026)
### Submissão 1 — Grupo de Trabalho nº 6

**Integrantes do Grupo:**
- Estanislau da Fonseca
- Maria da Conceição
- Maurício Piquito
- Ulundo Aurélio Moisés

---

### Objetivo:
Classificação dos 100 textos não rotulados de `dataset3_inputs.csv` através dos dois melhores modelos apurados no benchmark do projeto:
1. **Submissão 1 (s1):** PyTorch BiLSTM com Embeddings (TOP #1: F1 = 66.67%, ROC-AUC = 0.8333)
2. **Submissão 2 (s2):** PyTorch Tabular DNN com TF-IDF (TOP #2: F1 = 62.50%, ROC-AUC = 0.8095)

Os ficheiros CSV gerados respeitam estritamente a nomenclatura exigida:
- `submissao1-grupo6-s1.csv`
- `submissao1-grupo6-s2.csv`
com a coluna de saídas rotulada exclusivamente com `Human` ou `AI`.

In [ ]:
# 1. Configuração do ambiente e imports
import os
import sys
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Adicionar raiz e src ao sys.path
CURRENT_DIR = os.path.abspath("")
PROJECT_ROOT = os.path.abspath(os.path.join(CURRENT_DIR, "..")) if not os.path.exists("models") else CURRENT_DIR
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Dispositivo ativo: {device}")
print(f"[*] Diretório de modelos: {MODELS_DIR}")

In [ ]:
# 2. Definição das arquiteturas e des-serialização robusta
class SimpleTokenizer:
    def __init__(self, max_vocab_size=10000, max_seq_len=150):
        self.max_vocab_size = max_vocab_size
        self.max_seq_len = max_seq_len
        self.vocab = {"<PAD>": 0, "<UNK>": 1}

    def encode(self, text):
        tokens = str(text).lower().strip().split()
        indices = [self.vocab.get(token, self.vocab["<UNK>"]) for token in tokens]
        if len(indices) > self.max_seq_len:
            indices = indices[:self.max_seq_len]
        else:
            indices = indices + [self.vocab["<PAD>"]] * (self.max_seq_len - len(indices))
        return indices

    @property
    def vocab_size(self):
        return len(self.vocab)

class TextBiLSTM(nn.Module):
    def __init__(self, vocab_size=10000, embed_dim=128, hidden_dim=64, num_layers=2, dropout_rate=0.3):
        super(TextBiLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embed_dropout = nn.Dropout(dropout_rate)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0.0
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        embedded = self.embed_dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embedded)
        pooled, _ = torch.max(lstm_out, dim=1)
        return self.fc(pooled)

class TabularDNN(nn.Module):
    def __init__(self, input_dim=1000, hidden_dim1=128, hidden_dim2=64, dropout_rate=0.3):
        super(TabularDNN, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim1),
            nn.BatchNorm1d(hidden_dim1),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.BatchNorm1d(hidden_dim2),
            nn.ReLU(),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(hidden_dim2, 1)
        )

    def forward(self, x):
        return self.network(x)

class RobustUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        if name == 'SimpleTokenizer':
            return SimpleTokenizer
        if module == '__main__':
            if name in globals():
                return globals()[name]
        return super().find_class(module, name)

def carregar_pickle(path):
    with open(path, "rb") as f:
        return RobustUnpickler(f).load()

print("[+] Classes e carregadores registados com sucesso.")

In [ ]:
# 3. Carregamento do dataset3_inputs.csv
candidatos = [
    os.path.join(PROJECT_ROOT, "dataset3_inputs.csv"),
    os.path.join(DATA_DIR, "dataset3_inputs.csv"),
    os.path.join(CURRENT_DIR, "dataset3_inputs.csv")
]
caminho_csv = next((c for c in candidatos if os.path.exists(c)), None)
if caminho_csv is None:
    raise FileNotFoundError("dataset3_inputs.csv não encontrado!")

df_raw = pd.read_csv(caminho_csv, sep=";", encoding="utf-8-sig")
df_raw.columns = [c.strip().lstrip("\ufeff") for c in df_raw.columns]

ids = df_raw["ID"].astype(str).tolist()
texts = df_raw["Text"].astype(str).tolist()

print(f"[+] Dataset carregado: {len(texts)} amostras (IDs: {ids[0]} a {ids[-1]})")
display(df_raw.head())

In [ ]:
# 4. Modelo 1 (s1): PyTorch BiLSTM (Embeddings)
tokenizer = carregar_pickle(os.path.join(MODELS_DIR, "simple_tokenizer.pkl"))
model_bilstm = TextBiLSTM(vocab_size=tokenizer.vocab_size).to(device)
model_bilstm.load_state_dict(torch.load(os.path.join(MODELS_DIR, "best_pytorch_bilstm.pt"), map_location=device))
model_bilstm.eval()

encoded = [tokenizer.encode(t) for t in texts]
x_tensor_bilstm = torch.tensor(encoded, dtype=torch.long).to(device)

with torch.no_grad():
    probs_s1 = torch.sigmoid(model_bilstm(x_tensor_bilstm)).cpu().numpy().flatten()

labels_s1 = ["AI" if p >= 0.5 else "Human" for p in probs_s1]

df_s1 = pd.DataFrame({"ID": ids, "Output": labels_s1})
df_s1.to_csv("submissao1-grupo6-s1.csv", index=False)

print("[+] Submissão 1 (s1 - BiLSTM) guardada em 'submissao1-grupo6-s1.csv'")
print(f"    Distribuição: Human = {labels_s1.count('Human')}, AI = {labels_s1.count('AI')}")
display(df_s1.head())

In [ ]:
# 5. Modelo 2 (s2): PyTorch Tabular DNN (TF-IDF)
vectorizer = carregar_pickle(os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl"))
input_dim = len(vectorizer.get_feature_names_out())
model_dnn = TabularDNN(input_dim=input_dim).to(device)
model_dnn.load_state_dict(torch.load(os.path.join(MODELS_DIR, "best_pytorch_dnn.pt"), map_location=device))
model_dnn.eval()

x_tfidf = vectorizer.transform(texts).toarray()
x_tensor_dnn = torch.tensor(x_tfidf, dtype=torch.float32).to(device)

with torch.no_grad():
    probs_s2 = torch.sigmoid(model_dnn(x_tensor_dnn)).cpu().numpy().flatten()

labels_s2 = ["AI" if p >= 0.5 else "Human" for p in probs_s2]

df_s2 = pd.DataFrame({"ID": ids, "Output": labels_s2})
df_s2.to_csv("submissao1-grupo6-s2.csv", index=False)

print("[+] Submissão 2 (s2 - Tabular DNN) guardada em 'submissao1-grupo6-s2.csv'")
print(f"    Distribuição: Human = {labels_s2.count('Human')}, AI = {labels_s2.count('AI')}")
display(df_s2.head())

In [ ]:
# 6. Concordância entre os dois modelos
concordancia = sum(1 for a, b in zip(labels_s1, labels_s2) if a == b)
print(f"Concordância: {concordancia}/{len(ids)} ({concordancia/len(ids)*100:.1f}%)")